In [1]:
!pip install -q \
transformers \
sentence-transformers \
chromadb \
pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
import torch
import chromadb
import fitz

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Environment Ready")

Environment Ready


In [3]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded))

print(pdf_path)

Saving Transformer_Math_and_RAG_Explained.pdf to Transformer_Math_and_RAG_Explained.pdf
Transformer_Math_and_RAG_Explained.pdf


In [4]:
document = fitz.open(pdf_path)

print("Pages :", len(document))

Pages : 6


In [5]:
text = ""

for page in document:
    text += page.get_text()

In [6]:
import re

clean_text = re.sub(r"\s+", " ", text)

clean_text = clean_text.strip()

print(len(clean_text))

9837


In [7]:
chunk_size = 500

chunks = []

for i in range(0, len(clean_text), chunk_size):

    chunks.append(
        clean_text[i:i+chunk_size]
    )

In [8]:
print(type(chunks))

print()

print("Chunks :", len(chunks))

<class 'list'>

Chunks : 20


In [9]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding Model Ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Ready


In [10]:
chunk_embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
print(chunk_embeddings.shape)

(20, 384)


In [12]:
client = chromadb.Client()

In [13]:
collection = client.create_collection(
    name="enterprise_documents"
)

In [14]:
ids = []

for i in range(len(chunks)):
    ids.append(f"chunk_{i}")

In [15]:
collection.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)

print("Knowledge Stored")

Knowledge Stored


In [16]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Qwen Ready")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen Ready


In [18]:
question = "How do transformers convert words into vectors?"

print(question)

How do transformers convert words into vectors?


In [19]:
question_embedding = embedding_model.encode(question)

print(question_embedding.shape)

(384,)


In [20]:
results = collection.query(
    query_embeddings=[
        question_embedding.tolist()
    ],
    n_results=3
)

In [21]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [22]:
for i, doc in enumerate(results["documents"][0]):

    print("="*60)

    print(f"Retrieved Chunk {i}")

    print("="*60)

    print(doc)

    print()

Retrieved Chunk 0
w and expensive. Retrieval-Augmented Generation solves this without touching the model at all — it changes only what the model is shown, right before it starts predicting the next token. Document Embeddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-v

Retrieved Chunk 1
1(h0) → h(1) → ... → Layer_24(h23) → h(24) Technical detail Each transformer layer is a function that maps a vector of dimension d (e.g. 896) to another vector of the same dimension. Stacking 24 such layers progressively enriches the representation while preserving its shape, allowing contextual information to accumulate across the sequence. Step 4 — Attention (Deciding What Matters) In plain terms This is the core trick. Each word asks: "of everything else 

In [23]:
retrieved_chunks = results["documents"][0]

print(type(retrieved_chunks))
print()
print(len(retrieved_chunks))

<class 'list'>

3


In [24]:
context = "\n\n".join(retrieved_chunks)

print(type(context))
print()

print(len(context))

<class 'str'>

1504


In [25]:
print("=" * 60)

print(context[:1000])

print("=" * 60)

w and expensive. Retrieval-Augmented Generation solves this without touching the model at all — it changes only what the model is shown, right before it starts predicting the next token. Document Embeddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-v

1(h0) → h(1) → ... → Layer_24(h23) → h(24) Technical detail Each transformer layer is a function that maps a vector of dimension d (e.g. 896) to another vector of the same dimension. Stacking 24 such layers progressively enriches the representation while preserving its shape, allowing contextual information to accumulate across the sequence. Step 4 — Attention (Deciding What Matters) In plain terms This is the core trick. Each word asks: "of everything else in this sentence, what should I pay 

In [26]:
prompt = f"""
Use the following enterprise knowledge to answer the question.

Knowledge:
{context}

Question:
{question}

Answer:
"""

In [27]:
print(prompt[:1200])


Use the following enterprise knowledge to answer the question.

Knowledge:
w and expensive. Retrieval-Augmented Generation solves this without touching the model at all — it changes only what the model is shown, right before it starts predicting the next token. Document Embeddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-v

1(h0) → h(1) → ... → Layer_24(h23) → h(24) Technical detail Each transformer layer is a function that maps a vector of dimension d (e.g. 896) to another vector of the same dimension. Stacking 24 such layers progressively enriches the representation while preserving its shape, allowing contextual information to accumulate across the sequence. Step 4 — Attention (Deciding What Matters) In plain terms This is the core tri

In [28]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

print(inputs.keys())

KeysView({'input_ids': tensor([[  198, 10253,   279,  2701, 20179,  6540,   311,  4226,   279,  3405,
           382, 80334,   510,    86,   323, 11392,    13, 19470,   831, 61635,
         26980, 23470, 67477,   419,  2041, 30587,   279,  1614,   518,   678,
          1959,   432,  4344,  1172,  1128,   279,  1614,   374,  6839,    11,
          1290,  1573,   432,  8471, 51897,   279,  1790,  3950,    13, 11789,
         37068, 24602,  1959,  1032,   970,   279, 25739,  4149,   758, 14396,
          3793,  4599,  1075,  3842,  4244,   633,  6519,  1119,  7290,  8273,
         10605,    11,  4453,  9293,   320,   269, 26757,   315,  1105,     8,
           633,   862,  1828,  7290,  8273, 10605,  2238,    13,   362,  3405,
          5221, 16099,   279,  1852,  1616,    13,  5005,   279,  1849,  4936,
         12341,   892,  2197,  8273, 10605,  2444, 18093,   311,   279,  3405,
          8273,   271,    16,  3203,    15,     8, 11397,   305,     7,    16,
             8, 11397,  2503,

In [29]:
print(inputs["input_ids"].shape)

torch.Size([1, 357])


In [30]:
print("Number of Tokens")

print(inputs["input_ids"].shape[1])

Number of Tokens
357


In [31]:
with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=150
    )

In [32]:
answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(answer)


Use the following enterprise knowledge to answer the question.

Knowledge:
w and expensive. Retrieval-Augmented Generation solves this without touching the model at all — it changes only what the model is shown, right before it starts predicting the next token. Document Embeddings — Reusing the Same Math In plain terms Just like individual words get turned into meaning-vectors, entire documents (or chunks of them) get their own meaning-vectors too. A question gets converted the same way. Then the system simply checks which document-vectors sit closest to the question-v

1(h0) → h(1) → ... → Layer_24(h23) → h(24) Technical detail Each transformer layer is a function that maps a vector of dimension d (e.g. 896) to another vector of the same dimension. Stacking 24 such layers progressively enriches the representation while preserving its shape, allowing contextual information to accumulate across the sequence. Step 4 — Attention (Deciding What Matters) In plain terms This is the core tri